In [1]:
import torch
from trl import DPOTrainer, DPOConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Quantization: Load the model in 4-bit to save massive VRAM
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # The 3080 Ti supports bfloat16 natively
)

# 2. LoRA: Only train a tiny 1-2% fraction of the model's weights
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# 3. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Load the base model with the 4-bit config attached
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# 5. DPO Config with aggressive memory optimizations
training_args = DPOConfig(
    output_dir="./qwen-dpo-qlora",
    per_device_train_batch_size=1,   
    gradient_accumulation_steps=8,   
    gradient_checkpointing=True,     
    optim="paged_adamw_8bit",        
    max_length=1024,                 # Keeps the total sequence under 1024 tokens
    precompute_ref_log_probs=True,   # CRITICAL: Drops the reference model from VRAM!
    logging_steps=10,
    learning_rate=2e-4
)

# 6. Initialize Trainer
trainer = DPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=load_dataset("trl-lib/ultrafeedback_binarized"),
    peft_config=peft_config,         # Pass the LoRA config here!
)

# 7. Start training!
trainer.train()

/home/talos-1/miniforge3/envs/torch_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 1000/1000 [00:00<00:00, 5815.03 examples/s]


AttributeError: 'str' object has no attribute 'keys'